![alt text](./Cerny_logo_1.jpg)

# Analysis of Cerny ventilation recordings

#### Manual revision and trimming of ventilator recordings.

This notebook imports the preprocessed **Fabian ventilator parameters** data from pickle archive and performs trimming (removing periods when the patient was not connected to ventilator) based on manual inspection of data.

- At least 5 minutes of untrimmed ventilator and clinical data are available for **819 cases**
- At least 10 minutes of recording with **mechanical ventilation** is available after trimming in **407 cases**

The data processed and analysed in this Notebook were collected by the **Neonatal Emergency and Transport Service of the Peter Cerny Foundation**, Budapest, Hungary

**Author: Dr Gusztav Belteki**

### 1. Import the required libraries and set options

In [1]:
import IPython
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt

import os
import sys
import pickle

from pandas import Series, DataFrame
from datetime import datetime, timedelta
from matplotlib import dates

%matplotlib inline
matplotlib.style.use('classic')
matplotlib.rcParams['figure.facecolor'] = 'w'

pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 100)
# pd.set_option('mode.chained_assignment', None)

import warnings
warnings.simplefilter("ignore")

In [2]:
print("Python version: {}".format(sys.version))
print("pandas version: {}".format(pd.__version__))
print("matplotlib version: {}".format(matplotlib.__version__))
print("NumPy version: {}".format(np.__version__))
print("IPython version: {}".format(IPython.__version__))

Python version: 3.12.11 | packaged by conda-forge | (main, Jun  4 2025, 14:38:53) [Clang 18.1.8 ]
pandas version: 2.3.2
matplotlib version: 3.10.6
NumPy version: 1.26.4
IPython version: 9.5.0


### 2. List and set the working directory and the directory to write out data

In [4]:
# Name of the external hard drive
DRIVE = 'GB_1'

# Path to project folder containing ventilation research results
PATH = os.path.join(os.sep, 'Users', 'guszti', 'Library', 'Mobile Documents', 'com~apple~CloudDocs', 
                            'Documents', 'Research', 'Ventilation')

# Folder to export the result of analysis
DIR_WRITE = os.path.join(PATH, 'ventilation_fabian', 'Analyses', 'analysis_1_1100')
os.makedirs(DIR_WRITE, exist_ok = True)

# Folder on a USB stick to export data to and to import processed data exported by other Notebooks
DATA_DUMP = os.path.join(os.sep, 'Volumes', DRIVE, 'data_dump', 'fabian',)
os.makedirs(DATA_DUMP, exist_ok = True)

DIR_WRITE, DATA_DUMP

('/Users/guszti/Library/Mobile Documents/com~apple~CloudDocs/Documents/Research/Ventilation/ventilation_fabian/Analyses/analysis_1_1100',
 '/Volumes/GB_1/data_dump/fabian')

### 3. Import ventilator data from pickle archives

In [6]:
with open(os.path.join(DATA_DUMP, 'data_pars_measurements_1_1100.pickle'), 'rb') as handle:
    data_pars_measurements = pickle.load(handle)   
with open(os.path.join(DATA_DUMP, 'data_pars_settings_1_1100.pickle'), 'rb') as handle:
    data_pars_settings = pickle.load(handle)  
with open(os.path.join(DATA_DUMP, 'data_pars_alarms_1_1100.pickle'), 'rb') as handle:
    data_pars_alarms = pickle.load(handle)
len(data_pars_measurements)

819

### 4. Exploratory analysis on ventilation modes

##### How many cases of the different ventilation modes occur

In [9]:
vent_modes = {}
for case in data_pars_settings:
    # Multiply by two to get the number of seconds
    vent_modes[case] = data_pars_settings[case]['Ventilator_mode'].value_counts() * 2
vent_modes = DataFrame(vent_modes).T
vent_modes.replace(np.nan, 0, inplace = True)

In [13]:
# Add the duration of the recordings
recording_duration = {}
for case in data_pars_settings:
    recording_duration[case] = 2 * len(data_pars_settings[case])

In [14]:
vent_modes['ventilation'] = vent_modes['IPPV'] + vent_modes['SIMV'] + vent_modes['SIPPV'] + vent_modes['SIMVPSV']
vent_modes['noninvasive'] = vent_modes['CPAP'] + vent_modes['DUOPAP'] + vent_modes['NCPAP'] + vent_modes['O2therapy']  
vent_modes['total'] = Series(recording_duration)

In [15]:
print('SIMV:', sum(vent_modes['SIMV'] > 0))
print('SIPPV:', sum(vent_modes['SIPPV'] > 0))
print('SIMVPSV:', sum(vent_modes['SIMVPSV'] > 0))
print('PSV:', sum(vent_modes['PSV'] > 0))
print('IPPV:', sum(vent_modes['IPPV'] > 0))
print('NCPAP:', sum(vent_modes['NCPAP'] > 0))
print('CPAP:', sum(vent_modes['CPAP'] > 0))
print('DUOPAP:', sum(vent_modes['DUOPAP'] > 0))
print('O2therapy:', sum(vent_modes['O2therapy'] > 0))
print('ventilation:', sum(vent_modes['ventilation'] > 0))
print('noninvasive:', sum(vent_modes['noninvasive'] > 0))
print('total', len(vent_modes))

SIMV: 435
SIPPV: 255
SIMVPSV: 69
PSV: 4
IPPV: 95
NCPAP: 450
CPAP: 16
DUOPAP: 106
O2therapy: 132
ventilation: 625
noninvasive: 590
total 819


In [16]:
# How many seconds of each ventilation mode in total?
total_duration = DataFrame(vent_modes.sum(axis = 0), columns = ['duration (seconds)'])
total_duration

,duration (seconds)
Ventilator_mode,
CPAP,3634.0
DUOPAP,199362.0
IPPV,61136.0
NCPAP,1024542.0
O2therapy,271090.0
PSV,264.0
SIMV,1199432.0
SIMVPSV,100132.0
SIPPV,636658.0


##### Export Dataframes containing ventilator modes to Excel files and pickle archives

In [18]:
writer = pd.ExcelWriter(os.path.join(DIR_WRITE, 'ventilation_modes_1_1100.xlsx'))
vent_modes.to_excel(writer, 'vent_modes')
total_duration.to_excel(writer, 'total_duration')
writer.close()

In [19]:
with open(os.path.join(DATA_DUMP, 'vent_modes_1_1100'), 'wb') as handle:
    pickle.dump(vent_modes, handle, protocol=pickle.HIGHEST_PROTOCOL)

### 5. Only consider those recordings that have at least 10 minutes (600 seconds) mechanical ventilation

In [21]:
# Sampling rate is 0.5 Hz
vent_modes_ventilated = vent_modes[vent_modes['ventilation'] >= 600]
len(vent_modes_ventilated)

418

In [39]:
vent_modes_ventilated.head(100)

Ventilator_mode,CPAP,DUOPAP,IPPV,NCPAP,O2therapy,PSV,SIMV,SIMVPSV,SIPPV,ventilation,noninvasive,total
AL000003,0.0,0.0,0.0,4.0,0.0,0.0,6294.0,0.0,0.0,6294.0,4.0,6298
AL000006,0.0,0.0,0.0,164.0,0.0,0.0,8.0,2834.0,0.0,2842.0,164.0,3006
AL000007,0.0,0.0,0.0,0.0,0.0,0.0,7156.0,74.0,0.0,7230.0,0.0,7230
AL000008,0.0,0.0,0.0,0.0,0.0,0.0,6762.0,0.0,0.0,6762.0,0.0,6762
AL000009,0.0,0.0,0.0,0.0,0.0,0.0,42.0,0.0,2756.0,2798.0,0.0,2798
AL000011,0.0,0.0,0.0,16.0,0.0,0.0,5778.0,0.0,0.0,5778.0,16.0,5794
AL000014,0.0,0.0,0.0,2.0,0.0,0.0,3636.0,0.0,0.0,3636.0,2.0,3638
AL000018,0.0,0.0,0.0,120.0,0.0,0.0,0.0,2708.0,0.0,2708.0,120.0,2828
AL000019,0.0,0.0,0.0,0.0,0.0,0.0,2758.0,14.0,0.0,2772.0,0.0,2772
AL000020,0.0,0.0,0.0,0.0,0.0,0.0,204.0,0.0,7860.0,8064.0,0.0,8064


### 6. Remove the periods from the beginning and the end of the recordings when the patient was not connected to the ventilator

This requires manual inspection of the tidal volume and pressure graphs

This dictionary contains tuples of the start and end points as strings
This was obtained by manual inspection of VTmand and PIP and the recordings
and manually removing the start and the end when the baby was not on the ventilator (e.g. no VTmand)

In [35]:
with open(os.path.join(DATA_DUMP, 'limit_1_1100_ventilated.pickle'), 'rb') as handle:
    limit = pickle.load(handle)

In [37]:
limit

{'AL000064': ('2017-06-13 18:46:30', '2017-06-13 19:29:00'),
 'AL000003': ('2017-03-24 18:22:00', '2017-03-24 19:48:00'),
 'AL000114': ('2017-08-03 09:13:00', '2017-08-03 10:50:00'),
 'AL000285': ('2018-06-29 15:54:00', '2018-06-29 16:32:30'),
 'AL000118': ('2017-08-08 09:01:00', '2017-08-08 10:44:00'),
 'AL000268': ('2018-06-14 10:39:00', '2018-06-14 12:09:00'),
 'AL000090': ('2017-07-13 10:19:00', '2017-07-13 12:09:29'),
 'AL000018': ('2017-04-13 15:55:00', '2017-04-13 16:32:00'),
 'AL000246': ('2018-04-11 19:24:00', '2018-04-11 21:25:00'),
 'AL000035': ('2017-04-27 09:03:00', '2017-04-27 09:41:00'),
 'AL000284': ('2018-06-29 12:26:30', '2018-06-29 13:07:30'),
 'AL000215': ('2018-03-19 21:36:00', '2018-03-19 22:09:00'),
 'AL000279': ('2018-06-24 11:17:00', '2018-06-24 13:04:00'),
 'AL000176': ('2017-12-10 23:00:00', '2017-12-11 00:03:00'),
 'AL000067': ('2017-06-14 22:29:00', '2017-06-15 00:45:00'),
 'AL000048': ('2017-05-09 05:50:00', '2017-05-09 07:24:00'),
 'AL000250': ('2018-04-1

In [41]:
len(limit)

418

In [153]:
with open(os.path.join(DATA_DUMP, 'limit_1_1100_ventilated.pickle'), 'wb') as handle:
    pickle.dump(limit, handle, protocol=pickle.HIGHEST_PROTOCOL)

### 7. Trim ventilator data using the manual filters

In [156]:
# Trim ventilator data using the manual filters
data_pars_measurements_ventilated = {}
data_pars_settings_ventilated = {}
data_pars_alarms_ventilated = {}

for case in vent_modes_ventilated.index:
    data_pars_measurements_ventilated[case] = data_pars_measurements[case][limit[case][0] : limit[case][1]]
    data_pars_settings_ventilated[case] = data_pars_settings[case][limit[case][0] : limit[case][1]]
    data_pars_alarms_ventilated[case] = data_pars_alarms[case][limit[case][0] : limit[case][1]]

### 8. Remove periods which stayed in some recordings when the ventilation was stopped

In [159]:
data_pars_settings_ventilated['AL000428'][data_pars_settings_ventilated['AL000428']['Ventilation_stopped'] == 'yes'];

In [161]:
data_pars_measurements_ventilated['AL000428'].loc['2018-12-04 12:46:01': '2018-12-04 12:55:35'];

In [163]:
for case in data_pars_settings_ventilated:
    dta = data_pars_settings_ventilated[case]
    dta = dta[dta['Ventilation_stopped'] == 'no']
    data_pars_measurements_ventilated[case] = data_pars_measurements_ventilated[case].reindex(dta.index)
    data_pars_settings_ventilated[case] = data_pars_settings_ventilated[case].reindex(dta.index)
    data_pars_alarms_ventilated[case] = data_pars_alarms_ventilated[case].reindex(dta.index)

### 9. Remove any noninvasive parts accidentally remaining in the recording after manual trimming

In [165]:
mech_vent_modes = ['IPPV', 'SIPPV', 'SIMV', 'SIMVPSV', 'PSV']

for case in data_pars_settings_ventilated:
    dta = data_pars_settings_ventilated[case]
    dta = dta[dta['Ventilator_mode'].isin(mech_vent_modes)]
    data_pars_settings_ventilated[case] = dta

In [168]:
for case in data_pars_settings_ventilated:
    data_pars_measurements_ventilated[case] = data_pars_measurements_ventilated[case].reindex(data_pars_settings_ventilated[case].index)
    data_pars_alarms_ventilated[case] = data_pars_alarms_ventilated[case].reindex(data_pars_settings_ventilated[case].index)

### 10. Reanalyse the trimmed data as above

##### How many cases of the different ventilation modes occur?

In [172]:
vent_modes_ventilated = {}
for case in data_pars_settings_ventilated:
    # Multiply by two to get the number of seconds
    vent_modes_ventilated[case] = data_pars_settings_ventilated[case]['Ventilator_mode'].value_counts() * 2
    
vent_modes_ventilated = DataFrame(vent_modes_ventilated).T
vent_modes_ventilated.replace(np.nan, 0, inplace = True)

In [174]:
# Add VG data
VG = {}
for case in data_pars_settings_ventilated:
    try:
        # Multiply by two to get the number of seconds
        VG[case] = data_pars_settings_ventilated[case]['VG_state'].value_counts() * 2
    except KeyError:
        VG[case] = np.zeros(1)
        # print('No VG_state for %s' % case)
        
VG = DataFrame(VG).T
VG.columns = ['VG_off','VG_on']

vent_modes_ventilated = pd.concat([vent_modes_ventilated, VG], axis = 1)

In [176]:
# Add the total duration of the recordings
recording_duration_ventilated = {}
for case in data_pars_settings_ventilated:
    recording_duration_ventilated[case] = 2 * len(data_pars_settings_ventilated[case])

vent_modes_ventilated['total'] = Series(recording_duration_ventilated)
vent_modes_ventilated.sort_values('total')

,IPPV,PSV,SIMV,SIMVPSV,SIPPV,VG_off,VG_on,total
AL000629,0.0,0.0,0.0,0.0,0.0,NaN,NaN,0
AL000949,0.0,0.0,0.0,0.0,0.0,NaN,NaN,0
AL000518,0.0,0.0,0.0,0.0,0.0,NaN,NaN,0
AL000217,0.0,0.0,2.0,0.0,0.0,2.0,NaN,2
AL000492,0.0,0.0,4.0,0.0,0.0,4.0,NaN,4
...,...,...,...,...,...,...,...,...
AL000503,0.0,0.0,11580.0,0.0,0.0,11072.0,508.0,11580
AL000026,0.0,0.0,12122.0,0.0,0.0,12122.0,NaN,12122
AL000342,0.0,0.0,12900.0,0.0,0.0,12900.0,NaN,12900
AL000113,0.0,0.0,14208.0,0.0,0.0,10.0,14198.0,14208


### 11. Only keep recordings that have at least 10 minutes (600 seconds) mechanical ventilation after manual trimming

In [179]:
vent_modes_ventilated = vent_modes_ventilated[vent_modes_ventilated['total'] >= 600]
cases = sorted(vent_modes_ventilated.index)
len(vent_modes_ventilated), len(cases)

(407, 407)

In [181]:
# How many seconds of each ventilation mode in total?
total_duration_ventilated = DataFrame(vent_modes_ventilated.sum(axis = 0), columns = ['duration (seconds)'])
total_duration_ventilated

,duration (seconds)
IPPV,47704.0
PSV,208.0
SIMV,1042124.0
SIMVPSV,82824.0
SIPPV,554390.0
VG_off,475686.0
VG_on,1251564.0
total,1727250.0


In [183]:
print('SIMV:', sum(vent_modes_ventilated['SIMV'] > 0))
print('SIPPV:', sum(vent_modes_ventilated['SIPPV'] > 0))
print('SIMVPSV:', sum(vent_modes_ventilated['SIMVPSV'] > 0))
print('PSV:', sum(vent_modes_ventilated['PSV'] > 0))
print('IPPV:', sum(vent_modes_ventilated['IPPV'] > 0))
print('VG_on:', sum(vent_modes_ventilated['VG_on'] > 0))
print('total', len(vent_modes_ventilated))

SIMV: 261
SIPPV: 157
SIMVPSV: 29
PSV: 1
IPPV: 23
VG_on: 315
total 407


In [185]:
data_pars_measurements_ventilated = {rec : data_pars_measurements_ventilated[rec] for rec 
                                     in data_pars_measurements_ventilated if rec in vent_modes_ventilated.index}
data_pars_settings_ventilated = {rec : data_pars_settings_ventilated[rec] for rec 
                                     in data_pars_settings_ventilated if rec in vent_modes_ventilated.index}
data_pars_alarms_ventilated = {rec : data_pars_alarms_ventilated[rec] for rec 
                                     in data_pars_alarms_ventilated if rec in vent_modes_ventilated.index}
len(data_pars_measurements_ventilated), len(data_pars_settings_ventilated), len(data_pars_alarms_ventilated)

(407, 407, 407)

### 12. Export trimmed ventilator data as pickle archives

In [188]:
with open(os.path.join(DATA_DUMP, 'data_pars_measurements_ventilated_1_1100.pickle'), 'wb') as handle:
    pickle.dump(data_pars_measurements_ventilated, handle, protocol=pickle.HIGHEST_PROTOCOL)
with open(os.path.join(DATA_DUMP, 'data_pars_settings_ventilated_1_1100.pickle'), 'wb') as handle:
    pickle.dump(data_pars_settings_ventilated, handle, protocol=pickle.HIGHEST_PROTOCOL)
with open(os.path.join(DATA_DUMP, 'data_pars_alarms_ventilated_1_1100.pickle'), 'wb') as handle:
    pickle.dump(data_pars_alarms_ventilated, handle, protocol=pickle.HIGHEST_PROTOCOL)

### 13. Export data about ventilator modes to Excel files and pickle archives

In [191]:
writer = pd.ExcelWriter(os.path.join(DIR_WRITE, 'ventilation_modes_ventilated_1_1100.xlsx'))
vent_modes_ventilated.to_excel(writer, 'vent_modes_ventilated_1_1100')
total_duration_ventilated.to_excel(writer, 'total_duration_vent_1_1100')
writer.close()

In [193]:
with open(os.path.join(DATA_DUMP, 'vent_modes_ventilated_1_1100.pickle'), 'wb') as handle:
    pickle.dump(vent_modes_ventilated, handle, protocol=pickle.HIGHEST_PROTOCOL)